In [ ]:
from google.colab import drive
from pathlib import Path
from pyspark.sql import SparkSession

drive.mount('/content/drive')

spark = (
    SparkSession.builder
    .appName("CloudProviderAnalytics_Silver")
    .master("local[*]")
    .getOrCreate()
)

BRONZE_PATH = Path(
    "/content/drive/MyDrive/cloud-provider-analytics-data/bronze"
)

print("Version de Spark:", spark.version)
print("Existe Bronze:", BRONZE_PATH.exists())


Mounted at /content/drive
Version de Spark: 4.0.5
Existe Bronze: True


In [ ]:
customers = spark.read.parquet(
    str(BRONZE_PATH / "customers_orgs")
)

users = spark.read.parquet(
    str(BRONZE_PATH / "users")
)

billing = spark.read.parquet(
    str(BRONZE_PATH / "billing_monthly")
)

events = spark.read.parquet(
    str(BRONZE_PATH / "usage_events_reprocesado")
)

print("Customers:", customers.count())
print("Users:", users.count())
print("Billing:", billing.count())
print("Eventos:", events.count())

print("\nEsquema de eventos:")
events.printSchema()


Customers: 80
Users: 800
Billing: 240
Eventos: 43200

Esquema de eventos:
root
 |-- event_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- org_id: string (nullable = true)
 |-- resource_id: string (nullable = true)
 |-- service: string (nullable = true)
 |-- region: string (nullable = true)
 |-- metric: string (nullable = true)
 |-- value_raw: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- cost_usd_increment: double (nullable = true)
 |-- schema_version: integer (nullable = true)
 |-- carbon_kg: double (nullable = true)
 |-- genai_tokens: double (nullable = true)
 |-- value: double (nullable = true)
 |-- ingest_ts: timestamp (nullable = false)
 |-- source_file: string (nullable = false)



In [ ]:
from pyspark.sql import functions as F

eventos_revisados = events.withColumn(
    "motivo_calidad",
    F.concat_ws(
        " | ",
        F.when(F.col("event_id").isNull(), "event_id nulo"),
        F.when(F.col("cost_usd_increment") < -0.01, "costo inferior a -0,01"),
        F.when(
            F.col("value").isNotNull() & F.col("unit").isNull(),
            "valor sin unidad",
        ),
    ),
)

eventos_cuarentena = eventos_revisados.filter(F.col("motivo_calidad") != "")

eventos_cuarentena.groupBy("motivo_calidad").count().show(truncate=False)

+-----------------------------------------+-----+
|motivo_calidad                           |count|
+-----------------------------------------+-----+
|valor sin unidad                         |2033 |
|costo inferior a -0,01                   |206  |
|costo inferior a -0,01 | valor sin unidad|5    |
+-----------------------------------------+-----+



In [ ]:
RUTA_CUARENTENA_EVENTOS = BRONZE_PATH.parent / "quarantine" / "usage_events"

(
    eventos_cuarentena
    .withColumn("quarantine_ts", F.current_timestamp())
    .repartition(2)
    .write.mode("overwrite")
    .parquet(str(RUTA_CUARENTENA_EVENTOS))
)

cuarentena_guardada = spark.read.parquet(str(RUTA_CUARENTENA_EVENTOS))

print("Ruta:", RUTA_CUARENTENA_EVENTOS)
print("Eventos leídos desde la cuarentena:", cuarentena_guardada.count())

cuarentena_guardada.select(
    "event_id", "value", "unit", "cost_usd_increment",
    "motivo_calidad", "source_file",
).show(5, truncate=False)

Ruta: /content/drive/MyDrive/cloud-provider-analytics-data/quarantine/usage_events
Eventos leídos desde la cuarentena: 2244
+----------------+------+----+------------------+----------------+--------------------------------------------------------------------------------------------------------------+
|event_id        |value |unit|cost_usd_increment|motivo_calidad  |source_file                                                                                                   |
+----------------+------+----+------------------+----------------+--------------------------------------------------------------------------------------------------------------+
|evt_6dvzrbdq2mwb|11.813|NULL|0.1384            |valor sin unidad|file:///content/drive/MyDrive/cloud-provider-analytics-data/landing/usage_events_stream/events_part_0036.jsonl|
|evt_utkryur437p2|109.0 |NULL|1.0489            |valor sin unidad|file:///content/drive/MyDrive/cloud-provider-analytics-data/landing/usage_events_stream/events_par

In [ ]:
from pyspark.sql import functions as F

print("Total de organizaciones:", customers.count())

print(
    "NPS nulos:",
    customers.filter(F.col("nps_score").isNull()).count()
)

nps_invalidos = customers.filter(
    F.col("nps_score").isNotNull()
    & ~F.col("nps_score").between(-100, 100)
)

print(
    "NPS fuera del rango -100 a 100:",
    nps_invalidos.count()
)

print("\nRegistros con NPS fuera de rango:")
nps_invalidos.select(
    "org_id",
    "org_name",
    "nps_score"
).show(truncate=False)


Total de organizaciones: 80
NPS nulos: 11
NPS fuera del rango -100 a 100: 1

Registros con NPS fuera de rango:
+------------+--------------+---------+
|org_id      |org_name      |nps_score|
+------------+--------------+---------+
|org_pac56t4u|Gamma Cloud 37|101.0    |
+------------+--------------+---------+



In [ ]:
from pyspark.sql import functions as F

# Identificamos los NPS fuera del rango permitido
condicion_nps_invalido = (
    F.col("nps_score").isNotNull()
    & ~F.col("nps_score").between(-100, 100)
)

# Organizaciones que cumplen la regla de NPS
customers_silver = customers.filter(
    ~condicion_nps_invalido
)

# Registros que necesitan revisión
customers_quarantine = (
    customers
    .filter(condicion_nps_invalido)
    .withColumn(
        "quality_error",
        F.lit("nps_score fuera del rango -100 a 100")
    )
    .withColumn(
        "quarantine_ts",
        F.current_timestamp()
    )
)

print("Organizaciones en Bronze:", customers.count())
print("Organizaciones en Silver:", customers_silver.count())
print("Organizaciones en Quarantine:", customers_quarantine.count())

customers_quarantine.select(
    "org_id",
    "org_name",
    "nps_score",
    "quality_error"
).show(truncate=False)


Organizaciones en Bronze: 80
Organizaciones en Silver: 79
Organizaciones en Quarantine: 1
+------------+--------------+---------+------------------------------------+
|org_id      |org_name      |nps_score|quality_error                       |
+------------+--------------+---------+------------------------------------+
|org_pac56t4u|Gamma Cloud 37|101.0    |nps_score fuera del rango -100 a 100|
+------------+--------------+---------+------------------------------------+



In [ ]:
from pathlib import Path

DATA_PATH = BRONZE_PATH.parent

SILVER_PATH = DATA_PATH / "silver"
QUARANTINE_PATH = DATA_PATH / "quarantine"

ruta_customers_silver = SILVER_PATH / "customers_orgs"
ruta_customers_quarantine = QUARANTINE_PATH / "customers_orgs"

# Guardamos las organizaciones válidas
(
    customers_silver
    .write
    .mode("overwrite")
    .parquet(str(ruta_customers_silver))
)

# Guardamos los registros inválidos con su motivo
(
    customers_quarantine
    .write
    .mode("overwrite")
    .parquet(str(ruta_customers_quarantine))
)

print("Silver guardado en:", ruta_customers_silver)
print("Quarantine guardado en:", ruta_customers_quarantine)


Silver guardado en: /content/drive/MyDrive/cloud-provider-analytics-data/silver/customers_orgs
Quarantine guardado en: /content/drive/MyDrive/cloud-provider-analytics-data/quarantine/customers_orgs


In [ ]:
customers_silver_check = spark.read.parquet(
    str(ruta_customers_silver)
)

customers_quarantine_check = spark.read.parquet(
    str(ruta_customers_quarantine)
)

print("Registros en Silver:", customers_silver_check.count())
print("Registros en Quarantine:", customers_quarantine_check.count())

print(
    "Total recuperado:",
    customers_silver_check.count()
    + customers_quarantine_check.count()
)

customers_quarantine_check.select(
    "org_id",
    "org_name",
    "nps_score",
    "quality_error"
).show(truncate=False)



Registros en Silver: 79
Registros en Quarantine: 1
Total recuperado: 80
+------------+--------------+---------+------------------------------------+
|org_id      |org_name      |nps_score|quality_error                       |
+------------+--------------+---------+------------------------------------+
|org_pac56t4u|Gamma Cloud 37|101.0    |nps_score fuera del rango -100 a 100|
+------------+--------------+---------+------------------------------------+



In [ ]:
from pyspark.sql import functions as F

print("Total de usuarios:", users.count())

print(
    "user_id nulos:",
    users.filter(F.col("user_id").isNull()).count()
)

print(
    "user_id duplicados:",
    users.count()
    - users.select("user_id").distinct().count()
)

print(
    "last_login nulos:",
    users.filter(F.col("last_login").isNull()).count()
)

# Verificamos que cada usuario pertenezca
# a una organización existente en Bronze
orgs_validas = customers.select("org_id").distinct()

users_sin_org = users.join(
    orgs_validas,
    on="org_id",
    how="left_anti"
)

print(
    "Usuarios con org_id inexistente:",
    users_sin_org.count()
)


Total de usuarios: 800
user_id nulos: 0
user_id duplicados: 0
last_login nulos: 139
Usuarios con org_id inexistente: 0


In [ ]:
from pyspark.sql import functions as F

# Preparamos los usuarios para Silver
users_silver = (
    users
    .withColumn("silver_processed_ts", F.current_timestamp())
)

# Definimos la ruta de salida
ruta_users_silver = SILVER_PATH / "users"

# Guardamos los datos en Parquet
(
    users_silver
    .write
    .mode("overwrite")
    .parquet(str(ruta_users_silver))
)

print("Users Silver guardado en:", ruta_users_silver)


Users Silver guardado en: /content/drive/MyDrive/cloud-provider-analytics-data/silver/users


In [ ]:
users_silver_check = spark.read.parquet(
    str(ruta_users_silver)
)

print("Registros en Bronze:", users.count())
print("Registros en Silver:", users_silver_check.count())

print(
    "last_login nulos en Silver:",
    users_silver_check.filter(
        F.col("last_login").isNull()
    ).count()
)

users_silver_check.select(
    "user_id",
    "org_id",
    "active",
    "created_at",
    "last_login",
    "silver_processed_ts"
).show(5, truncate=False)


Registros en Bronze: 800
Registros en Silver: 800
last_login nulos en Silver: 139
+-------------+------------+------+----------+----------+--------------------------+
|user_id      |org_id      |active|created_at|last_login|silver_processed_ts       |
+-------------+------------+------+----------+----------+--------------------------+
|user_9tze5r5u|org_3t60rjiw|true  |2025-07-10|NULL      |2026-11-12 20:41:54.878048|
|user_qpgb7r3o|org_afeyuhz1|true  |2025-07-15|2025-08-10|2026-11-12 20:41:54.878048|
|user_cwbfuk9e|org_ykl1cq99|true  |2025-06-12|2025-07-26|2026-11-12 20:41:54.878048|
|user_1v2isqp4|org_g8sbi4q2|false |2025-05-12|NULL      |2026-11-12 20:41:54.878048|
|user_9kwv08hh|org_pht0hl9x|true  |2025-08-01|2025-08-11|2026-11-12 20:41:54.878048|
+-------------+------------+------+----------+----------+--------------------------+
only showing top 5 rows


In [ ]:
from pyspark.sql import functions as F

print("Total de facturas:", billing.count())

print(
    "invoice_id nulos:",
    billing.filter(F.col("invoice_id").isNull()).count()
)

print(
    "invoice_id duplicados:",
    billing.count()
    - billing.select("invoice_id").distinct().count()
)

print(
    "credits nulos:",
    billing.filter(F.col("credits").isNull()).count()
)

print(
    "subtotal negativos:",
    billing.filter(F.col("subtotal") < 0).count()
)

print(
    "taxes negativos:",
    billing.filter(F.col("taxes") < 0).count()
)

print(
    "Tipos de cambio nulos o no positivos:",
    billing.filter(
        F.col("exchange_rate_to_usd").isNull()
        | (F.col("exchange_rate_to_usd") <= 0)
    ).count()
)

# Validamos la relación con las organizaciones originales
billing_sin_org = billing.join(
    customers.select("org_id").distinct(),
    on="org_id",
    how="left_anti"
)

print(
    "Facturas con org_id inexistente:",
    billing_sin_org.count()
)


Total de facturas: 240
invoice_id nulos: 0
invoice_id duplicados: 0
credits nulos: 137
subtotal negativos: 13
taxes negativos: 0
Tipos de cambio nulos o no positivos: 0
Facturas con org_id inexistente: 0


In [ ]:
from pyspark.sql import functions as F

# Identificamos las facturas con subtotal negativo
billing_negativos = billing.filter(
    F.col("subtotal") < 0
)

print(
    "Cantidad de facturas con subtotal negativo:",
    billing_negativos.count()
)

# Mostramos sus principales campos
billing_negativos.select(
    "invoice_id",
    "org_id",
    "month",
    "subtotal",
    "credits",
    "taxes",
    "currency",
    "exchange_rate_to_usd"
).orderBy("subtotal").show(20, truncate=False)

# Revisamos cómo se distribuyen por moneda
print("\nFacturas negativas por moneda:")

billing_negativos.groupBy("currency").count().show()


Cantidad de facturas con subtotal negativo: 13
+------------+------------+----------+--------+-------+------+--------+--------------------+
|invoice_id  |org_id      |month     |subtotal|credits|taxes |currency|exchange_rate_to_usd|
+------------+------------+----------+--------+-------+------+--------+--------------------+
|inv_jwe46xaa|org_afeyuhz1|2025-06-01|-1671.83|3.60   |351.08|USD     |1.01834000          |
|inv_0kg6umh4|org_ktakpuxq|2025-08-01|-1355.74|NULL   |284.71|ARS     |0.00157000          |
|inv_gx8uwtk7|org_g0fn9xuy|2025-08-01|-1125.54|3.42   |236.36|USD     |1.00132000          |
|inv_sy80w5gs|org_ly8ozcyw|2025-08-01|-1095.10|NULL   |229.97|EUR     |0.99810000          |
|inv_9rvn97y8|org_vyie6ivw|2025-08-01|-910.68 |70.98  |191.24|USD     |0.96732000          |
|inv_3tlo1i7x|org_x7eedtjv|2025-08-01|-879.97 |56.93  |184.79|USD     |0.95074000          |
|inv_bmzw7539|org_i7p5tb94|2025-08-01|-849.43 |18.20  |178.38|USD     |1.04212000          |
|inv_b17zyqzs|org_nam14

In [ ]:
from pyspark.sql import functions as F

negativas = billing.filter(F.col("subtotal") < 0).alias("neg")
positivas = billing.filter(F.col("subtotal") > 0).alias("pos")

posibles_anulaciones = negativas.join(
    positivas,
    (
        (F.col("neg.org_id") == F.col("pos.org_id"))
        & (F.col("neg.currency") == F.col("pos.currency"))
        & (F.col("neg.subtotal") == -F.col("pos.subtotal"))
    ),
    "inner"
)

posibles_anulaciones.select(
    F.col("neg.invoice_id").alias("factura_negativa"),
    F.col("pos.invoice_id").alias("factura_positiva"),
    F.col("neg.org_id").alias("org_id"),
    F.col("neg.subtotal").alias("importe_negativo"),
    F.col("pos.subtotal").alias("importe_positivo"),
    F.col("neg.currency").alias("moneda"),
    F.col("neg.month").alias("mes_negativo"),
    F.col("pos.month").alias("mes_positivo")
).show(30, truncate=False)

print(
    "Cantidad de posibles coincidencias:",
    posibles_anulaciones.count()
)


+----------------+----------------+------+----------------+----------------+------+------------+------------+
|factura_negativa|factura_positiva|org_id|importe_negativo|importe_positivo|moneda|mes_negativo|mes_positivo|
+----------------+----------------+------+----------------+----------------+------+------------+------------+
+----------------+----------------+------+----------------+----------------+------+------------+------------+

Cantidad de posibles coincidencias: 0


In [ ]:
from pyspark.sql import functions as F

# Conservamos todas las facturas y marcamos
# los subtotales negativos para revisión
billing_silver = (
    billing
    .withColumn(
        "requires_review",
        F.col("subtotal") < 0
    )
    .withColumn(
        "silver_processed_ts",
        F.current_timestamp()
    )
)

ruta_billing_silver = SILVER_PATH / "billing_monthly"

(
    billing_silver
    .write
    .mode("overwrite")
    .parquet(str(ruta_billing_silver))
)

print("Billing Silver guardado en:", ruta_billing_silver)


Billing Silver guardado en: /content/drive/MyDrive/cloud-provider-analytics-data/silver/billing_monthly


In [ ]:
billing_silver_check = spark.read.parquet(
    str(ruta_billing_silver)
)

print("Registros en Bronze:", billing.count())
print("Registros en Silver:", billing_silver_check.count())

print(
    "Facturas marcadas para revisión:",
    billing_silver_check.filter(
        F.col("requires_review")
    ).count()
)

print(
    "Créditos nulos conservados:",
    billing_silver_check.filter(
        F.col("credits").isNull()
    ).count()
)


Registros en Bronze: 240
Registros en Silver: 240
Facturas marcadas para revisión: 13
Créditos nulos conservados: 137


In [ ]:
from pyspark.sql import functions as F

print("Total de eventos:", events.count())

print(
    "event_id nulos:",
    events.filter(F.col("event_id").isNull()).count()
)

print(
    "event_id duplicados:",
    events.count()
    - events.select("event_id").distinct().count()
)

print(
    "cost_usd_increment negativos:",
    events.filter(
        F.col("cost_usd_increment") < 0
    ).count()
)

print(
    "value nulos:",
    events.filter(F.col("value").isNull()).count()
)

print(
    "unit nulos:",
    events.filter(F.col("unit").isNull()).count()
)

print("\nEventos por versión de esquema:")
events.groupBy("schema_version").count().orderBy("schema_version").show()

# Validamos las referencias a organizaciones
events_sin_org = events.join(
    customers.select("org_id").distinct(),
    on="org_id",
    how="left_anti"
)

print(
    "Eventos con org_id inexistente:",
    events_sin_org.count()
)


Total de eventos: 43200
event_id nulos: 0
event_id duplicados: 0
cost_usd_increment negativos: 216
value nulos: 2186
unit nulos: 2075

Eventos por versión de esquema:
+--------------+-----+
|schema_version|count|
+--------------+-----+
|             1|10800|
|             2|32400|
+--------------+-----+

Eventos con org_id inexistente: 0


In [ ]:
from pyspark.sql import functions as F

# Analizamos los nulos y costos negativos
# según la versión del esquema de eventos

events.groupBy("schema_version").agg(
    F.count("*").alias("total_eventos"),

    F.sum(
        F.when(F.col("value").isNull(), 1).otherwise(0)
    ).alias("value_nulos"),

    F.sum(
        F.when(F.col("unit").isNull(), 1).otherwise(0)
    ).alias("unit_nulos"),

    F.sum(
        F.when(
            F.col("cost_usd_increment") < 0, 1
        ).otherwise(0)
    ).alias("costos_negativos"),

    F.sum(
        F.when(F.col("carbon_kg").isNull(), 1).otherwise(0)
    ).alias("carbon_kg_nulos"),

    F.sum(
        F.when(F.col("genai_tokens").isNull(), 1).otherwise(0)
    ).alias("genai_tokens_nulos")

).orderBy("schema_version").show(truncate=False)


+--------------+-------------+-----------+----------+----------------+---------------+------------------+
|schema_version|total_eventos|value_nulos|unit_nulos|costos_negativos|carbon_kg_nulos|genai_tokens_nulos|
+--------------+-------------+-----------+----------+----------------+---------------+------------------+
|1             |10800        |544        |546       |62              |10800          |10800             |
|2             |32400        |1642       |1529      |154             |0              |29268             |
+--------------+-------------+-----------+----------+----------------+---------------+------------------+



In [ ]:
from pyspark.sql import functions as F

events.groupBy("service").agg(
    F.count("*").alias("total_eventos"),

    F.sum(
        F.when(F.col("value").isNull(), 1).otherwise(0)
    ).alias("value_nulos"),

    F.sum(
        F.when(F.col("unit").isNull(), 1).otherwise(0)
    ).alias("unit_nulos"),

    F.sum(
        F.when(
            F.col("cost_usd_increment") < 0, 1
        ).otherwise(0)
    ).alias("costos_negativos"),

    F.sum(
        F.when(
            F.col("genai_tokens").isNotNull(), 1
        ).otherwise(0)
    ).alias("genai_tokens_informados")

).orderBy(F.desc("total_eventos")).show(
    30, truncate=False
)


+----------+-------------+-----------+----------+----------------+-----------------------+
|service   |total_eventos|value_nulos|unit_nulos|costos_negativos|genai_tokens_informados|
+----------+-------------+-----------+----------+----------------+-----------------------+
|compute   |12498        |637        |626       |56              |0                      |
|storage   |7614         |382        |363       |44              |0                      |
|database  |7419         |402        |360       |44              |0                      |
|networking|6921         |332        |318       |32              |0                      |
|analytics |4590         |215        |205       |19              |0                      |
|genai     |4158         |218        |203       |21              |3132                   |
+----------+-------------+-----------+----------+----------------+-----------------------+



In [ ]:
from pyspark.sql import functions as F

events_silver = (
    eventos_revisados
    .filter(F.col("event_id").isNotNull())
    .withColumn("event_date", F.to_date("timestamp"))
    .withColumn(
        "missing_usage_data",
        F.col("value").isNull() | F.col("unit").isNull(),
    )
    .withColumn("negative_cost_flag", F.col("cost_usd_increment") < 0)
    .withColumn("cost_anomaly_flag", F.col("cost_usd_increment") < -0.01)
    .withColumn(
        "missing_genai_tokens",
        (F.col("schema_version") == 2)
        & (F.col("service") == "genai")
        & F.col("genai_tokens").isNull(),
    )
    .withColumn(
        "requests",
        F.when(
            (F.col("metric") == "requests") & (F.col("unit") == "count"),
            F.col("value"),
        ),
    )
    .withColumn("silver_processed_ts", F.current_timestamp())
)

events_silver.agg(
    F.count("*").alias("eventos_silver"),
    F.count_if("cost_anomaly_flag").alias("costos_bajo_umbral"),
    F.count("requests").alias("eventos_con_requests_validos"),
).show()

+--------------+------------------+----------------------------+
|eventos_silver|costos_bajo_umbral|eventos_con_requests_validos|
+--------------+------------------+----------------------------+
|         43200|               211|                       18215|
+--------------+------------------+----------------------------+



In [ ]:
dim_organizaciones = customers.select(
    "org_id", "org_name", "industry", "hq_region", "plan_tier"
)

events_silver_enriquecido = events_silver.join(
    dim_organizaciones,
    on="org_id",
    how="left",
)

events_silver_enriquecido.agg(
    F.count("*").alias("eventos_enriquecidos"),
    F.countDistinct("org_id").alias("organizaciones"),
    F.count_if(F.col("org_name").isNull()).alias("eventos_sin_nombre_org"),
).show()

+--------------------+--------------+----------------------+
|eventos_enriquecidos|organizaciones|eventos_sin_nombre_org|
+--------------------+--------------+----------------------+
|               43200|            80|                     0|
+--------------------+--------------+----------------------+



In [ ]:
ruta_events_silver = BRONZE_PATH.parent / "silver" / "usage_events"

(
    events_silver_enriquecido
    .repartition(4)
    .write.mode("overwrite")
    .parquet(str(ruta_events_silver))
)

events_silver_guardado = spark.read.parquet(str(ruta_events_silver))

print("Ruta Silver:", ruta_events_silver)

events_silver_guardado.agg(
    F.count("*").alias("eventos"),
    F.countDistinct("org_id").alias("organizaciones"),
    F.count("requests").alias("eventos_con_requests_validos"),
    F.round(F.sum("cost_usd_increment"), 4).alias("costo_total_usd"),
).show()

Ruta Silver: /content/drive/MyDrive/cloud-provider-analytics-data/silver/usage_events
+-------+--------------+----------------------------+---------------+
|eventos|organizaciones|eventos_con_requests_validos|costo_total_usd|
+-------+--------------+----------------------------+---------------+
|  43200|            80|                       18215|    147433.9778|
+-------+--------------+----------------------------+---------------+



In [ ]:
events_silver_check = spark.read.parquet(
    str(ruta_events_silver)
)

print(
    "Eventos en Bronze:",
    events.count()
)

print(
    "Eventos en Silver:",
    events_silver_check.count()
)

print(
    "Datos de uso incompletos:",
    events_silver_check.filter(
        F.col("missing_usage_data")
    ).count()
)

print(
    "Costos negativos:",
    events_silver_check.filter(
        F.col("negative_cost_flag")
    ).count()
)

print(
    "GenAI sin tokens:",
    events_silver_check.filter(
        F.col("missing_genai_tokens")
    ).count()
)

events_silver_check.select(
    "event_id",
    "service",
    "value",
    "unit",
    "cost_usd_increment",
    "missing_usage_data",
    "negative_cost_flag",
    "missing_genai_tokens"
).show(5, truncate=False)


Eventos en Bronze: 43200
Eventos en Silver: 43200
Datos de uso incompletos: 4164
Costos negativos: 216
GenAI sin tokens: 1026
+----------------+----------+------+--------+------------------+------------------+------------------+--------------------+
|event_id        |service   |value |unit    |cost_usd_increment|missing_usage_data|negative_cost_flag|missing_genai_tokens|
+----------------+----------+------+--------+------------------+------------------+------------------+--------------------+
|evt_xq8se8nco5kx|compute   |2.6787|hours   |0.2995            |false             |false             |false               |
|evt_9emyh5yry4i9|networking|0.0   |gb_hours|0.0               |false             |false             |false               |
|evt_resrgp6f7mjf|compute   |3.301 |hours   |0.2396            |false             |false             |false               |
|evt_1bzo9q5feymu|storage   |2.2324|hours   |0.6068            |false             |false             |false               |
|evt_5